<a href="https://colab.research.google.com/github/ML-Bioinfo-CEITEC/bioinfo-school/blob/main/exercises/week3/B_protein_embeddings_esm2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Embeddings with ESM-2


### Install & Import Libraries

In [ ]:
# Install necessary libraries
!pip install -qq transformers einops numpy pandas scikit-learn matplotlib seaborn umap-learn torch biopython

In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM, EsmModel
import einops
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import umap.umap_ as umap
import matplotlib.pyplot as plt
import seaborn as sns

### Load ESM-2 Model and Tokenizer

In [ ]:
# Load pre-trained ESM-2 model and tokenizer
model_name = "facebook/esm2_t6_8M_UR50D" # Smallest model for demonstration
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name)

## Embedding Extraction

### Load Protein Sequences

In [ ]:
from Bio import SeqIO

def load_fasta(fasta_file):
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequences.append(str(record.seq))
    return sequences

protein_sequences = load_fasta("/content/proteins.fasta")
print(f"Loaded {len(protein_sequences)} protein sequences.")
print("First sequence:", protein_sequences[0][:100], "...")

### Tokenize Sequences and Extract Embeddings

In [ ]:
def get_embeddings(sequences, tokenizer, model):
    all_per_residue_embeddings = []
    all_per_sequence_embeddings = []

    for seq in sequences:
        inputs = tokenizer(seq, return_tensors='pt', truncation=True, max_length=1024)
        with torch.no_grad():
            outputs = model(**inputs)

        # Per-residue embeddings (all hidden states)
        # The last hidden state is typically used for per-residue embeddings
        per_residue_embeddings = outputs.last_hidden_state.squeeze(0).cpu().numpy()
        all_per_residue_embeddings.append(per_residue_embeddings)

        # Per-sequence embedding (CLS token embedding)
        # The first token (CLS token) is typically used for sequence-level representation
        per_sequence_embedding = outputs.last_hidden_state[:, 0].squeeze(0).cpu().numpy()
        all_per_sequence_embeddings.append(per_sequence_embedding)

    return all_per_residue_embeddings, all_per_sequence_embeddings

per_residue_embeddings, per_sequence_embeddings = get_embeddings(protein_sequences, tokenizer, model)

print(f"Extracted {len(per_residue_embeddings)} sets of per-residue embeddings.")
print(f"Shape of first per-residue embedding: {per_residue_embeddings[0].shape}")
print(f"Extracted {len(per_sequence_embeddings)} per-sequence embeddings.")
print(f"Shape of first per-sequence embedding: {per_sequence_embeddings[0].shape}")

## Dimensionality Reduction and Clustering

### Load Protein Accessions (Metadata)

In [ ]:
import pandas as pd
# Load protein accessions to link with embeddings for better analysis
protein_accessions_df = pd.read_csv("/content/protein_accessions.tsv", sep="\t")
print(f"Loaded {len(protein_accessions_df)} protein accessions.")
print("First 5 accessions:\n", protein_accessions_df.head())

### UMAP Dimensionality Reduction

In [ ]:
import numpy as np
import umap.umap_ as umap

# Convert list of arrays to a single 2D NumPy array
per_sequence_embeddings_array = np.array(per_sequence_embeddings)

# UMAP
umapper = umap.UMAP(random_state=42)
umap_embeddings = umapper.fit_transform(per_sequence_embeddings_array)

print(f"UMAP embeddings shape: {umap_embeddings.shape}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add UMAP embeddings to the DataFrame for plotting
protein_accessions_df['umap_x'] = umap_embeddings[:, 0]
protein_accessions_df['umap_y'] = umap_embeddings[:, 1]

# Rename the '# family' column to 'family' for easier plotting
if '# family' in protein_accessions_df.columns:
    protein_accessions_df = protein_accessions_df.rename(columns={'# family': 'family'})

plt.figure(figsize=(10, 8))
sns.scatterplot(x='umap_x', y='umap_y', hue='family', data=protein_accessions_df, palette='tab10', s=70) # Changed palette to 'tab10'
plt.title('UMAP of Protein Embeddings by Family')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend(title='Protein Family', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### Mean-Pooling of Residue Embeddings

As sequence-level embeddings extracted by taking only the `<cls>` token (index 0) can sometimes be noisy or not fully representative, it is often better to compute the mean-pool of all non-special tokens in the sequence.

In [ ]:
# Compute mean-pooled embeddings (excluding special tokens <cls> at index 0 and <eos> at the end)
mean_pooled_embeddings = []
for res_emb in per_residue_embeddings:
    if len(res_emb) > 2:
        mean_emb = res_emb[1:-1].mean(axis=0)
    else:
        mean_emb = res_emb.mean(axis=0)
    mean_pooled_embeddings.append(mean_emb)
mean_pooled_embeddings = np.array(mean_pooled_embeddings)
print("Mean pooled embeddings shape:", mean_pooled_embeddings.shape)

### Pairwise Cosine Similarity

Compute the similarity matrix between all pairs of protein embeddings, plot the sorted similarity matrix as a heatmap, and compare average intra-family vs inter-family similarities.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute pairwise cosine similarity matrix
similarity_matrix = cosine_similarity(mean_pooled_embeddings)

# Get family and labels from the DataFrame
families = protein_accessions_df['family'].tolist()
labels = protein_accessions_df['label'].tolist()

# Group and sort by family to visualize the clusters on the heatmap
sorted_indices = np.argsort(families)
sorted_sim = similarity_matrix[sorted_indices][:, sorted_indices]
sorted_labels = [labels[idx] for idx in sorted_indices]

# Plot Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(sorted_sim, xticklabels=sorted_labels, yticklabels=sorted_labels, cmap="plasma", robust=True)
plt.title("Pairwise Cosine Similarity of ESM-2 Embeddings (Grouped by Family)")
plt.xlabel("Proteins")
plt.ylabel("Proteins")
plt.tight_layout()
plt.show()

# Calculate statistics
intra_sims = []
inter_sims = []
for i in range(len(families)):
    for j in range(i + 1, len(families)):
        sim = similarity_matrix[i, j]
        if families[i] == families[j]:
            intra_sims.append(sim)
        else:
            inter_sims.append(sim)

print(f"Average cosine similarity within the same family: {np.mean(intra_sims):.4f}")
print(f"Average cosine similarity between different families: {np.mean(inter_sims):.4f}")
print(f"Difference (Intra - Inter): {np.mean(intra_sims) - np.mean(inter_sims):.4f}")

### PCA Dimensionality Reduction

Perform Principal Component Analysis (PCA) on the mean-pooled embeddings to project them onto a 2D space and check if they cluster by family.

In [ ]:
from sklearn.decomposition import PCA

# Fit PCA
pca = PCA(n_components=2, random_state=42)
pca_embeddings = pca.fit_transform(mean_pooled_embeddings)

# Plot PCA
plt.figure(figsize=(10, 8))
sns.scatterplot(x=pca_embeddings[:, 0], y=pca_embeddings[:, 1], hue=families, palette='tab10', s=70)
plt.title('PCA of Protein Embeddings by Family')
plt.xlabel(f'PC 1 ({pca.explained_variance_ratio_[0]*100:.1f}% explained var)')
plt.ylabel(f'PC 2 ({pca.explained_variance_ratio_[1]*100:.1f}% explained var)')
plt.legend(title='Protein Family', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### Validation Check: Nearest Neighbor Analysis

Perform a simple validation check: check what percentage of proteins have their most similar partner (highest cosine similarity, excluding self) belonging to the same coarse family.

In [ ]:
correct_neighbors = 0
for i in range(len(families)):
    sims = similarity_matrix[i].copy()
    sims[i] = -1.0 # Exclude self
    nearest_idx = np.argmax(sims)
    if families[i] == families[nearest_idx]:
        correct_neighbors += 1

neighbor_acc = (correct_neighbors / len(families)) * 100
print(f"Nearest Neighbor Accuracy (same family): {neighbor_acc:.2f}% ({correct_neighbors}/{len(families)})")